In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nbformat  # Add nbformat import

# --- Synthetic IoT Data Generation ---

np.random.seed(42)
num_samples = 10000
t = np.linspace(0, 24*30, num_samples)  # simulate 30 days of hourly readings

# Latent factors: daily cycle, weekly cycle, device drift
daily_cycle = np.stack([np.sin(2 * np.pi * t / 24), np.cos(2 * np.pi * t / 24)], axis=1)
weekly_cycle = np.stack([np.sin(2 * np.pi * t / (24*7)), np.cos(2 * np.pi * t / (24*7))], axis=1)
device_drift = (t / (24*30))[:, None]  # linear drift over month

h3_true = np.concatenate([daily_cycle, weekly_cycle, device_drift], axis=1)  # (10000,5)

# True weight projections for two intermediate latent layers
input_dim = 6
h1_dim = 12
h2_dim = 8
top_dim = h3_true.shape[1]

W3 = np.random.randn(h2_dim, top_dim)
W2 = np.random.randn(h1_dim, h2_dim)
W1 = np.random.randn(input_dim, h1_dim)

# Generate hierarchical data: h3_true -> h2_true -> h1_true -> x_data
h2_true = h3_true @ W3.T + 0.1 * np.random.randn(num_samples, h2_dim)
h1_true = h2_true @ W2.T + 0.1 * np.random.randn(num_samples, h1_dim)
x_data = h1_true @ W1.T + 0.1 * np.random.randn(num_samples, input_dim)

# Inject anomalies in a segment
anomaly_indices = np.arange(2000, 2100)
x_data_anom = x_data.copy()
x_data_anom[anomaly_indices] += np.random.randn(len(anomaly_indices), input_dim) * 3

# --- Extended Predictive Coding Modules ---

class PredictiveCodingLayer:
    def __init__(self, input_dim, latent_dim):
        self.W = np.random.randn(input_dim, latent_dim) * 0.05

    def inference(self, x, iterations=20, lr_h=0.1):
        batch = x.shape[0]
        h = np.zeros((batch, self.W.shape[1]))
        for _ in range(iterations):
            x_hat = h @ self.W.T
            err = x - x_hat
            h += lr_h * (err @ self.W)
        return h, err

    def update(self, x, h, lr=1e-3):
        grad = x.T @ h / x.shape[0]
        self.W += lr * grad

class DeepPCNetwork:
    def __init__(self, dims):
        self.layers = [PredictiveCodingLayer(dims[i], dims[i+1]) for i in range(len(dims)-1)]

    def train(self, data, epochs=30):
        for epoch in range(epochs):
            h = data
            latents = [h]
            # Inference up the hierarchy
            for layer in self.layers:
                h, _ = layer.inference(h)
                latents.append(h)
            # Update weights top-down
            for i, layer in enumerate(self.layers):
                layer.update(latents[i], latents[i+1])

    def reconstruct(self, data):
        h = data
        for layer in self.layers:
            h, _ = layer.inference(h)
        # reconstruct back down
        for layer in reversed(self.layers):
            h = h @ layer.W.T
        return h

# Build and train deep PC network
dims = [input_dim, h1_dim, h2_dim, top_dim]
deep_net = DeepPCNetwork(dims)
deep_net.train(x_data_anom, epochs=50)

# Reconstruction
recon = deep_net.reconstruct(x_data_anom)

# Compute reconstruction error
errors = np.linalg.norm(x_data_anom - recon, axis=1)

# --- Visualization ---

# Create subplots
fig = make_subplots(rows=3, cols=1, subplot_titles=(
    'Sensor Channel 0: Original vs Reconstructed',
    'Reconstruction Error Over Time',
    'True vs Estimated Top Latent Dim 0'
))

# 1. Original vs Reconstructed for one sensor channel over a window
sensor_idx = 0
window = np.arange(1950, 2150)

fig.add_trace(
    go.Scatter(x=window, y=x_data_anom[window, sensor_idx], name='Original'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=window, y=recon[window, sensor_idx], name='Reconstructed'),
    row=1, col=1
)

# 2. Reconstruction Error Highlighting Anomalies
fig.add_trace(
    go.Scatter(x=np.arange(len(errors)), y=errors, name='Reconstruction Error'),
    row=2, col=1
)
fig.add_vrect(
    x0=2000, x1=2100,
    fillcolor="gray", opacity=0.3,
    layer="below", line_width=0,
    row=2, col=1
)

# 3. Latent Representation Scatter Plot (top layer)
_, top_est = deep_net.layers[-1].inference(
    deep_net.layers[-2].inference(
        deep_net.layers[-3].inference(x_data_anom)[0]
    )[0]
)

fig.add_trace(
    go.Scatter(
        x=h3_true[:,0],
        y=top_est[:,0],
        mode='markers',
        marker=dict(size=5),
        name='Latent Representation'
    ),
    row=3, col=1
)

# Update layout
fig.update_layout(
    height=1200,
    showlegend=True,
    title_text="Predictive Coding Analysis"
)

# Update axes labels
fig.update_xaxes(title_text="Sample Index", row=1, col=1)
fig.update_yaxes(title_text="Sensor Reading", row=1, col=1)
fig.update_xaxes(title_text="Sample Index", row=2, col=1)
fig.update_yaxes(title_text="Error Norm", row=2, col=1)
fig.update_xaxes(title_text="True h3_true[:,0]", row=3, col=1)
fig.update_yaxes(title_text="Estimated Top Latent", row=3, col=1)

# Save figure to HTML instead of showing directly
fig.write_html("predictive_coding_analysis.html")
